# 02 - Logistic Regression Model

This notebook trains a Logistic Regression baseline to classify student `performance_category` using the cleaned dataset from the preprocessing notebook.

The preprocessing and classifier are saved together as a scikit-learn pipeline in a `.pkl` file for later backend inference. Logistic Regression uses solver iterations rather than epochs.

In [1]:
from pathlib import Path
import pickle

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# Project paths
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True, parents=True)

DATA_PATH = PROCESSED_DIR / 'student_performance_processed.csv'
MODEL_PATH = MODEL_DIR / 'student_performance_model.pkl'

print(f'Loading dataset from: {DATA_PATH}')
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

Loading dataset from: /workspaces/LearnIQ---Personalized-Learning-Performance-Prediction-System/data/processed/student_performance_processed.csv
(5000, 20)


,student_id,age,grade_level,previous_marks,attendance_pct,study_hours_per_week,study_days_per_week,study_consistency_pct,sleep_hours_per_night,quiz_average,assignment_average,missed_assignments,average_attempts,study_frequency,learning_style,internet_access,final_marks,passed,performance_category,risk_level
0,STU04711,24.0,year 3,61.7,100.0,1.9,2,73.1,6.6,68.5,39.1,2,1,low,auditory,0.0,65.3,1.0,good,low
1,STU00692,17.0,year 4,45.4,78.7,0.1,1,65.2,7.4,47.2,46.5,2,3,low,practice,0.0,42.4,1.0,average,medium
2,STU00066,17.0,year 4,65.0,69.7,1.7,1,88.3,5.1,61.2,68.8,3,3,low,practice,0.0,62.4,1.0,good,low
3,STU03427,17.0,year 2,52.2,93.2,11.8,3,78.4,7.2,68.6,64.0,2,2,medium,visual,0.0,58.9,1.0,average,low
4,STU04821,23.0,year 2,71.5,80.8,7.7,3,97.1,7.5,72.1,55.1,1,3,medium,visual,0.0,68.0,1.0,good,low


In [3]:
# Define target and feature columns
# Exclude outcome-derived fields to avoid target leakage.
target_col = 'performance_category'
id_col = 'student_id'

if target_col not in df.columns:
    raise ValueError(f'Target column {target_col} not found in the processed dataset.')

excluded_cols = [id_col, target_col, 'risk_level', 'final_marks', 'passed']
X = df.drop(columns=excluded_cols, errors='ignore').copy()
y = df[target_col].copy()

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

Numeric features: ['age', 'previous_marks', 'attendance_pct', 'study_hours_per_week', 'study_days_per_week', 'study_consistency_pct', 'sleep_hours_per_night', 'quiz_average', 'assignment_average', 'missed_assignments', 'average_attempts', 'internet_access']
Categorical features: ['grade_level', 'study_frequency', 'learning_style']


In [4]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f'Train size: {len(X_train)}')
print(f'Test size: {len(X_test)}')

Train size: 4000
Test size: 1000


In [5]:
# Build preprocessing and model pipeline
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        # Logistic regression uses solver iterations, not neural-network epochs.
        ('classifier', LogisticRegression(max_iter=1500)),
    ]
)

# Train the baseline model
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Baseline model accuracy: {accuracy:.4f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred))
print('\nConfusion matrix:')
print(confusion_matrix(y_test, y_pred))

Baseline model accuracy: 0.6300

Classification report:
              precision    recall  f1-score   support

     average       0.66      0.74      0.70       417
   excellent       0.66      0.50      0.57       143
        good       0.59      0.62      0.61       369
        poor       0.53      0.32      0.40        71

    accuracy                           0.63      1000
   macro avg       0.61      0.54      0.57      1000
weighted avg       0.63      0.63      0.62      1000


Confusion matrix:
[[307   2  88  20]
 [  4  71  68   0]
 [105  35 229   0]
 [ 46   0   2  23]]


In [6]:
# Save the trained model pipeline for backend use
with open(MODEL_PATH, 'wb') as model_file:
    pickle.dump(model, model_file)

print(f'Model saved to: {MODEL_PATH}')
print('This .pkl file can be loaded later by the backend for inference.')

Model saved to: /workspaces/LearnIQ---Personalized-Learning-Performance-Prediction-System/models/student_performance_model.pkl
This .pkl file can be loaded later by the backend for inference.


## Conclusion

This notebook trains a Logistic Regression baseline to classify student `performance_category`. Outcome-derived fields such as `final_marks`, `passed`, and `risk_level` are excluded from the input features to reduce target leakage. The 1,500 setting is the solver's maximum iteration count, not an epoch count. Use the held-out test metrics above as a baseline when comparing future models, such as Random Forest or gradient boosting. The fitted preprocessing-and-classification pipeline is saved to `models/student_performance_model.pkl` for backend inference.